# 🎭 Namespaces, Scope & Decorators in Python
## 🎯 The Ultimate Masterclass

### 📖 The Story: How We Get Here
To master Decorators, we cannot just memorize the `@` syntax. We must follow a strict logical path to understand *why* Python works the way it does:
1. **Namespaces:** Where do variables actually live in memory?
2. **Scope:** Where are we allowed to look for them? (The LEGB Rule)
3. **Closures:** Can functions remember the environment they were born in?
4. **Decorators:** Can we use closures to wrap and supercharge functions?

### 🗺️ Notebook Roadmap
```mermaid
flowchart TD
A[1. Namespaces & Memory] --> B[2. Scope & LEGB Rule]
B --> C[3. Closures & Factories]
C --> D[4. Decorator Internals]
D --> E[5. Advanced & Stacking]
E --> F[6. Industry & Built-ins]
F --> G[7. Interview Cheat Sheet]
```

## 📦 1. Namespaces: The Storage of Python

### 📖 What is a Namespace?
A **Namespace** is exactly what it sounds like: a "space" that holds "names" (identifiers like variables, functions, and classes). Programmatically speaking, it is simply a **Python dictionary** under the hood:
* **Keys** = Variable/Function names (e.g., `x`, `my_list`, `calculate_sum`)
* **Values** = The actual objects in memory that those names point to (e.g., `5`, `[1, 2, 3]`, `<function calculate_sum>`).

| Namespace | Lifecycle | Example |
| :--- | :--- | :--- |
| **1. Built-in** | Created when Python starts. Dies when Python stops. | `print()`, `len()`, `ValueError` |
| **2. Global** | Created when a script/module starts. Dies when script ends. | Variables defined at the top level. |
| **3. Enclosing** | Created when an outer function is called. Dies when outer function ends. | Variables in the outer function. |
| **4. Local** | Created when a function is called. Dies when function returns. | Variables inside the function. |

### 🧠 Memory Visualization
When you write `x = 10`, Python doesn't just "create a variable". It adds an entry to a dictionary:

```text
Namespace Dictionary
┌─────────┬──────────────────┐
│ Name    │ Object in Memory │
├─────────┼──────────────────┤
│ 'x'     │ 10               │
│ 'name'  │ 'Anuj'           │
│ 'add'   │ <function add>   │
└─────────┴──────────────────┘
```
> **💡 Key Insight:** Variables are just labels (keys) pointing to objects (values).

> 💡 **Fun Fact:** Every time you create a variable, function, or class in Python, you're just adding an entry to some namespace dictionary! Python's `globals()` and `locals()` functions let you peek inside these dictionaries!


In [4]:
# ==========================================
# 📦 INSPECTING NAMESPACES IN PYTHON
# ==========================================

# 1. GLOBAL NAMESPACE
x = 10
name = "Anuj"
def add(a, b): return a + b

print("🌍 GLOBAL NAMESPACE (Sample):")
print(f"   'x' in globals(): {'x' in globals()}")
print(f"   'add' in globals(): {globals()['add']}")

🌍 GLOBAL NAMESPACE (Sample):
   'x' in globals(): True
   'add' in globals(): <function add at 0x000001844097D440>


In [2]:
# 2. LOCAL NAMESPACE
def my_function():
    local_var = "I am local"
    print("\n📦 LOCAL NAMESPACE (Inside function):")
    print(f"   Keys in locals(): {list(locals().keys())}")

my_function()


📦 LOCAL NAMESPACE (Inside function):
   Keys in locals(): ['local_var']


In [3]:
# 3. BUILT-IN NAMESPACE
import builtins
print("\n🏗️ BUILT-IN NAMESPACE (Sample):")
print(f"   'print' in builtins: {'print' in dir(builtins)}")
print(f"   'len' in builtins: {'len' in dir(builtins)}")
print(f"   Total built-in names: {len(dir(builtins))}")


🏗️ BUILT-IN NAMESPACE (Sample):
   'print' in builtins: True
   'len' in builtins: True
   Total built-in names: 161


## 🔭 2. Scope & The LEGB Rule

### 📖 What is Scope?
While a *namespace* is the dictionary storing the names, a **Scope** is a textual region of a Python program where a specific namespace is **directly accessible**.

### 📖 Namespace vs. Scope
| Concept | Definition | Analogy |
| :--- | :--- | :--- |
| **Namespace** | The actual dictionary mapping names to objects. | The Company Database (Storage) |
| **Scope** | The textual region where you can access that namespace. | Employee Clearance (Visibility) |

### 🧠 The LEGB Rule (Name Resolution)
When you type a variable name (e.g., `x`), how does Python know which `x` you mean? It searches the namespaces from the inside out using the **LEGB Rule**:

```mermaid
flowchart TD
A[Start Search for 'x'] --> B{1. Local Scope}
B -- "Found" --> C[Use Local 'x']
B -- "Not Found" --> D{2. Enclosing Scope}
D -- "Found" --> E[Use Enclosing 'x']
D -- "Not Found" --> F{3. Global Scope}
F -- "Found" --> G[Use Global 'x']
F -- "Not Found" --> H{4. Built-in Scope}
H -- "Found" --> I[Use Built-in 'x']
H -- "Not Found" --> J[💥 NameError]
```
> 💡 **Memory Trick:** **L**et's **E**at **G**ood **B**urgers.

In [5]:
# ==========================================
# 🔭 LEGB DEMONSTRATION & MODIFICATION
# ==========================================

x = "Global X"  # GLOBAL

def outer():
    x = "Enclosing X"  # ENCLOSING
    
    def inner():
        x = "Local X"  # LOCAL
        print(f"🔍 Inside inner(): {x}")
        
    inner()
    print(f"🔍 Inside outer(): {x}")

outer()
print(f"🔍 At module level: {x}")

🔍 Inside inner(): Local X
🔍 Inside outer(): Enclosing X
🔍 At module level: Global X


### Modifying Scopes (`global` vs `nonlocal`)
Reading globals is easy, but modifying them is different. If you try to update a global variable inside a function without telling Python, it will assume you are trying to create a *new* Local variable and throw an `UnboundLocalError`.

* **`global` keyword:** Tells Python: "Use the Global variable, don't create a local one."
* **`nonlocal` keyword:** Tells Python: "Use the Enclosing variable, don't create a local one."

In [12]:
# ==========================================
# 🛠️ MODIFYING SCOPES: global vs nonlocal
# ==========================================

count = 0

def bad_increment():
    # count += 1  # Uncommenting this causes UnboundLocalError!
    pass 

def global_increment():
    global count  
    count += 1

global_increment()
print(f"1️⃣ Using 'global': count is now {count}")

def make_counter():
    count = 10
    def increment():
        nonlocal count 
        count += 1
    increment()
    return count

print(f"2️⃣ Using 'nonlocal': count is now {make_counter()}")

# ==========================================
# ⚠️ DANGER: Shadowing Built-ins
# ==========================================
# len = 100  # Uncommenting this breaks len()!

1️⃣ Using 'global': count is now 1
2️⃣ Using 'nonlocal': count is now 11


## 🎁 3. Closures: Why Do They Exist?

Before we learn Decorators, we must understand **Closures**. 
To understand Closures, we must understand that **Functions are First-Class Citizens**.

### 🏆 What are First-Class Citizens?
In Python, functions are objects. This means you can:
1. Assign them to variables.
2. Pass them as arguments to other functions.
3. **Return them from other functions.**

### 🤐 What is a Closure?
A **Closure** is a nested function that **remembers** the variables from its enclosing scope, *even after the outer function has finished executing and returned*.



Most tutorials explain *what* a closure is. Few explain *why* it matters.

### 🤔 The Big Question
```python
def outer():
    count = 0
    def inner():
        return count
    return inner

func = outer()
print(func()) # Prints 0!
```
**Wait a minute.** `outer()` has finished executing. Its local namespace should have been destroyed. **Why does `inner()` still remember `count`?**

### 🎒 The "Backpack" Analogy
A closure is like a backpack. When a nested function is created, it packs up the variables it needs from the outer function and carries them wherever it goes. When a nested function is returned, it doesn't just leave empty-handed. It packs up the variables it needs from the enclosing scope into a **backpack** (the `__closure__` attribute) and carries them wherever it goes.

> 💡 **Why this matters:** **Decorators are just closures in disguise.** Without closures, decorators could not remember the original function they are wrapping!


In [13]:
# ==========================================
# 🎁 CLOSURES: FUNCTIONS WITH MEMORY
# ==========================================

def multiplier_factory(factor):
    """Outer function that creates a multiplier"""
    
    def multiplier(number):
        """Inner function (The Closure)"""
        # 'factor' is not local to multiplier(). 
        # It is remembered from the enclosing scope!
        return number * factor
        
    return multiplier

# Create specific functions using the factory
double = multiplier_factory(2)
triple = multiplier_factory(3)

print(f"double(5) = {double(5)}")  # Output: 10
print(f"triple(5) = {triple(5)}")  # Output: 15

# 🕵️‍♂️ PROOF: The closure remembers 'factor' even though multiplier_factory is done!
print(f"\n🎒 double's backpack (__closure__): {double.__closure__[0].cell_contents}")
print(f"🎒 triple's backpack (__closure__): {triple.__closure__[0].cell_contents}")

double(5) = 10
triple(5) = 15

🎒 double's backpack (__closure__): 2
🎒 triple's backpack (__closure__): 3


## 🎭 4. Decorators: Internals & Syntax

### 📖 Definition
A **Decorator** is a function that receives another function as input, **adds some functionality** (decoration) to it, and returns a modified function. It relies entirely on **First-Class Functions** and **Closures**.

### The Big Problem Decorators Solve
Imagine you have 50 functions in your app, and you want to track how long each one takes to run. 
* **Without decorators:** You have to paste `start_time` and `end_time` logging code inside all 50 functions. If you want to change how logging works later, you have to edit 50 places.
* **With decorators:** You write the logging logic ONCE inside a decorator, and simply "tag" the functions you want to track.

### 🧩 The `@` Syntactic Sugar

Instead of writing out `my_func = decorator(my_func)`, Python gives us the `@` symbol:

```python
@timer
def hello():
    pass
```
**What Python ACTUALLY does under the hood:**
```python
hello = timer(hello)
```

### 🔄 Decorator Internal Flow
```text
Step 1: timer(hello) is called.
Step 2: 'wrapper' closure is created (remembering 'hello').
Step 3: 'wrapper' is returned.
Step 4: The name 'hello' is reassigned to point to 'wrapper'.
```


```mermaid
flowchart LR
A[Original Function] --> B(Decorator Function)
B --> C[Wrapper Function]
C --> D[Adds Behavior Before]
D --> E[Calls Original Function]
E --> F[Adds Behavior After]
F --> G[Returns Modified Function]
```

In [8]:
# ==========================================
# 🎭 THE ULTIMATE PRODUCTION DECORATOR
# ==========================================
import functools
import time

def production_timer(func):
    """The Gold Standard Decorator Template"""
    
    # 1. PRESERVE METADATA (Crucial for debugging/Flask/FastAPI)
    @functools.wraps(func) 
    def wrapper(*args, **kwargs):  # 2. ACCEPT ANY ARGUMENTS
        start = time.time()
        
        result = func(*args, **kwargs)  # 3. CALL ORIGINAL
        
        end = time.time()
        print(f"⏱️ {func.__name__} took {end-start:.5f}s")
        return result  # 4. RETURN RESULT
        
    return wrapper

# ==========================================
# 🕵️‍♂️ THE functools.wraps DEEP DIVE
# ==========================================
def bad_decorator(func):
    def wrapper(*args, **kwargs): return func(*args, **kwargs)
    return wrapper

@bad_decorator
def my_func():
    """My docstring"""
    pass

print("❌ WITHOUT @wraps:")
print(f"   Name: {my_func.__name__}")  # Prints 'wrapper' (BAD!)
print(f"   Doc: {my_func.__doc__}")    # Prints None (BAD!)

@production_timer
def good_func():
    """My preserved docstring"""
    time.sleep(0.1)

print("\n✅ WITH @wraps:")
print(f"   Name: {good_func.__name__}")  # Prints 'good_func' (GOOD!)
print(f"   Doc: {good_func.__doc__}")    # Prints 'My preserved docstring'

good_func()

❌ WITHOUT @wraps:
   Name: wrapper
   Doc: None

✅ WITH @wraps:
   Name: good_func
   Doc: My preserved docstring
⏱️ good_func took 0.10068s


In [14]:
# ==========================================
# 🎭 YOUR FIRST DECORATOR
# ==========================================
import time

# 1. Define the Decorator
def timer_decorator(func):
    """A decorator that measures execution time"""
    
    def wrapper():
        print(f"⏳ Starting '{func.__name__}'...")
        start_time = time.time()
        
        # Call the actual original function
        func() 
        
        end_time = time.time()
        print(f"✅ Finished '{func.__name__}' in {end_time - start_time:.4f} seconds\n")
        
    return wrapper

# 2. Apply the Decorator (Manual Way)
def say_hello():
    print("   👋 Hello, World!")

decorated_hello = timer_decorator(say_hello)
decorated_hello()

# 3. Apply the Decorator (Syntactic Sugar Way)
@timer_decorator
def calculate_sum():
    print("   🧮 Calculating sum of 1 to 1,000,000...")
    sum(range(1_000_000))

calculate_sum() # Notice we just call it normally!

⏳ Starting 'say_hello'...
   👋 Hello, World!
✅ Finished 'say_hello' in 0.0000 seconds

⏳ Starting 'calculate_sum'...
   🧮 Calculating sum of 1 to 1,000,000...
✅ Finished 'calculate_sum' in 0.0130 seconds



## 🚀 5. Advanced Patterns

If we use the basic decorator above, we face **Two Massive Problems** in production:

### ❌ Problem 1: It breaks functions with arguments
If `func()` takes arguments (e.g., `func(a, b)`), our `wrapper()` will crash with a `TypeError`.
**Solution:** We use `*args` and `**kwargs` inside the wrapper so it can accept any inputs and pass them down.

### ❌ Problem 2: It destroys metadata (Introspection)
If we print `calculate_sum.__name__`, it will say `"wrapper"`, not `"calculate_sum"`. This breaks documentation and debugging tools.
**Solution:** We use the built-in `@functools.wraps` decorator on our wrapper.

### 🤯  Decorators with Arguments

What if we want to pass an argument to the decorator itself? (e.g., `@repeat(times=3)`). standard decorator only takes `func`.
To do this, we need **Three Levels of Nesting**. 
1. The Factory: Takes the decorator arguments and returns the decorator.
2. The Decorator: Takes the function and returns the wrapper.
3. The Wrapper: Executes the logic.

**Solution:** A **Decorator Factory** (3 levels of nesting).
```python
def retry(times):          # 1. Factory (takes args)
    def decorator(func):   # 2. Decorator (takes func)
        def wrapper(...):  # 3. Wrapper (takes func args)
```

### 🥞 Decorator Stacking Execution Flow
When you stack decorators, they wrap like an onion. 
```python
@A
@B
def func(): pass
```
**Execution Order when `func()` is called:**
```text
CALL func()
   ↓
A wrapper starts
   ↓
   B wrapper starts
      ↓
      Original function executes
      ↓
   B wrapper ends
   ↓
A wrapper ends
```


In [16]:
# ==========================================
# 🤯 DECORATORS WITH ARGUMENTS (e.g., @retry(times=3))
# ==========================================

def repeat(num_times):
    """A decorator factory that takes an argument"""
    
    def decorator_repeat(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(num_times):
                print(f"--- Execution {i+1} of {num_times} ---")
                result = func(*args, **kwargs)
            return result
        return wrapper
        
    return decorator_repeat  # Notice we return the DECORATOR, not the wrapper!

# Usage:
@repeat(num_times=3)
def greet(name):
    print(f"Hello {name}!")

greet("Anuj")

--- Execution 1 of 3 ---
Hello Anuj!
--- Execution 2 of 3 ---
Hello Anuj!
--- Execution 3 of 3 ---
Hello Anuj!


In [10]:
# ==========================================
# 🚀 DECORATOR FACTORY & STACKING
# ==========================================
import functools

# 1. Decorator Factory (Takes Arguments)
def repeat(num_times):
    def decorator_repeat(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator_repeat

# 2. Stacking Decorators
def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"**{func(*args, **kwargs)}**"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"*{func(*args, **kwargs)}*"
    return wrapper

@bold
@italic
@repeat(2)
def greet(name):
    return f"Hello {name}"

print("Stacking Execution:")
print(greet("Anuj")) 
# Output: ***Hello Anuj****Hello Anuj***

Stacking Execution:
***Hello Anuj***


In [15]:
# ==========================================
# 🚀 THE ULTIMATE PRODUCTION DECORATOR
# ==========================================
import functools

def production_logger(func):
    """
    The Gold Standard Decorator Template.
    Memorize this structure for interviews!
    """
    @functools.wraps(func)  # PRESERVES metadata (__name__, __doc__)
    def wrapper(*args, **kwargs):  # ACCEPTS any arguments
        print(f"🔍 [LOG] Calling: {func.__name__} with args={args}, kwargs={kwargs}")
        
        try:
            # Execute the actual function and capture result
            result = func(*args, **kwargs)
            print(f"✅ [LOG] {func.__name__} executed successfully.")
            return result
            
        except Exception as e:
            print(f"❌ [LOG] {func.__name__} failed with error: {e}")
            raise  # Re-raise the exception after logging
            
    return wrapper

# Let's test it with a function that takes arguments!
@production_logger
def divide(a, b):
    """Divides two numbers"""
    return a / b

print("Test: divide(10, 2)")
print(f"Result: {divide(10, 2)}\n")

# PROOF that metadata is preserved!
print(f"🕵️‍♂️ Function Name: {divide.__name__}")  # Should be 'divide', NOT 'wrapper'
print(f"🕵️‍♂️ Docstring: {divide.__doc__}")      # Should be 'Divides two numbers'

Test: divide(10, 2)
🔍 [LOG] Calling: divide with args=(10, 2), kwargs={}
✅ [LOG] divide executed successfully.
Result: 5.0

🕵️‍♂️ Function Name: divide
🕵️‍♂️ Docstring: Divides two numbers


## 🏛️ 6. Industry Standards & Built-in Decorators


While user-defined decorators are amazing, Python comes with powerful built-ins, especially for Object-Oriented Programming.

| Decorator | Purpose |
| :--- | :--- |
| **`@staticmethod`** | Utility Method, doesn't access instance (`self`) or class (`cls`) state. Acts like a plain function inside a class namespace. |
| **`@classmethod`** | Method receives the class (`cls`) as the first argument instead of instance (`self`). Great for alternative constructors. |
| **`@property`** | Turns a method into a getter/setter  for a read-only attribute. Allows adding logic (like validation) without changing the API. |
| **`@abstractmethod`** | Forces subclasses to implement a method (used with `ABC` module). |


### 🌐 Web Frameworks (The Real World)
* **Flask:** `@app.route("/")` maps URLs to functions.
* **FastAPI:** `@app.get("/users")` handles routing and OpenAPI docs.
* **Django:** `@login_required` handles authentication.

### 🧪 Testing & Standard Library
* **`@pytest.fixture`** sets up test environments.
* **`@functools.lru_cache`** memoizes expensive functions (Huge interview topic!).

In [11]:
# ==========================================
# 🌐 REAL-WORLD INDUSTRY EXAMPLES
# ==========================================
from functools import lru_cache

# 1. Caching / Memoization (Standard Library)
@lru_cache(maxsize=None)
def fibonacci(n):
    if n < 2: return n
    return fibonacci(n-1) + fibonacci(n-2)

print(f"🚀 Fast Fibonacci: {fibonacci(100)}")

# 2. Mocking Flask/FastAPI Routing
class MockApp:
    def __init__(self):
        self.routes = {}
        
    def route(self, path):
        def decorator(func):
            self.routes[path] = func
            return func
        return decorator

app = MockApp()

@app.route("/api/users")
def get_users():
    return ["Anuj", "Niketan"]

print(f"\n🌐 Flask-style Router:")
print(f"   Registered routes: {list(app.routes.keys())}")
print(f"   Calling /api/users: {app.routes['/api/users']()}")

# 3. OOP @property
class Circle:
    def __init__(self, radius):
        self._radius = radius
        
    @property
    def area(self):
        return 3.14 * self._radius ** 2

c = Circle(5)
print(f"\n🔵 @property (Computed Attribute): Area = {c.area}")

🚀 Fast Fibonacci: 354224848179261915075

🌐 Flask-style Router:
   Registered routes: ['/api/users']
   Calling /api/users: ['Anuj', 'Niketan']

🔵 @property (Computed Attribute): Area = 78.5


## 🎤 7. Interview Mastery & Cheat Sheet

### 📊 Quick Comparison Tables
  
| Namespace **vs** | Scope|
| :--- | :--- |
| **Namespace** | The Dictionary (Storage) |
| **Scope** | The Region of Code (Visibility) |

| `global` vs `nonlocal` | |
| :--- | :--- |
| **`global`** | Modifies Module-level variables |
| **`nonlocal`** | Modifies Enclosing (Parent function) variables |

| Concept | One-Liner Definition |
| :--- | :--- |
| **Closure** | A function + its remembered birth environment (backpack). |
| **Decorator** | A function that takes a function and returns a function. |
| **`@wraps`** | Copies metadata (`__name__`, `__doc__`) from original to wrapper. |

### 🏆 Top 3 FAANG Interview Questions
1. **Q: What happens if you don't use `@functools.wraps`?**
   *A: The decorated function loses its original `__name__` and `__doc__`, breaking debugging tools, introspection, and frameworks like Flask/FastAPI.*
2. **Q: How do you make a decorator accept arguments (e.g., `@retry(3)`)?**
   *A: You need 3 levels of nested functions: Factory (takes args) -> Decorator (takes func) -> Wrapper (takes func args).*
3. **Q: Explain the execution order of stacked decorators.**
   *A: They are applied **bottom-up** during definition (`A(B(func))`), but execute **top-down** when called.*

## 🎤 Master Interview Questions & Explanations

### 🟢 Beginner Level
**Q1: What is the difference between a Namespace and a Scope?**
> **Answer:** A **Namespace** is the actual dictionary mapping names to objects. A **Scope** is the *textual region* of code where you can directly access that namespace without a prefix.

**Q2: Explain the LEGB rule.**
> **Answer:** It defines the order Python searches for a variable: **L**ocal $\rightarrow$ **E**nclosing $\rightarrow$ **G**lobal $\rightarrow$ **B**uilt-in. Python searches from inside out and stops at the first match it finds.

**Q3: What is the `@` symbol?**
> **Answer:** It is syntactic sugar for decorators. `@my_decorator \n def func(): pass` is exactly equivalent to `func = my_decorator(func)`.

### 🟡 Intermediate Level
**Q4: What is a Closure?**
> **Answer:** A closure is a nested function that retains access to variables in its enclosing lexical scope, even after the outer function has completed execution. Decorators rely heavily on closures so the `wrapper` remembers the original `func`.

**Q5: What does `@functools.wraps` do and why is it important?**
> **Answer:** It copies the metadata (`__name__`, `__doc__`, `__module__`) from the original function to the wrapper function. Without it, debugging tools and documentation generators will see the generic name "wrapper" instead of the actual function name.

### 🔴 Advanced Level
**Q6: How do you create a decorator that takes arguments (e.g., `@retry(times=3)`)?**
> **Answer:** You need **three levels of nested functions**. The outermost function takes the decorator arguments and returns the actual decorator. The middle function takes the function to be decorated and returns the wrapper. The innermost wrapper executes the logic.

**Q7: What happens if you stack multiple decorators? In what order do they execute?**
> **Answer:** They are applied **bottom-up** (closest to the function first), but they execute **top-down** (outermost wrapper runs first). 

> `@A`

> `@B`

> `def func():`

> is mathematically equivalent to `func = A(B(func))`.

---

## 💡 Fun Facts

1. 🎂 **Age of the `@` Symbol:** The `@` syntax for decorators was introduced in Python 2.4 (released in 2004). Before that, developers had to write `func = decorator(func)` manually!
2. 📚 **Origin of the Name:** The term "Decorator" comes from the **Decorator Design Pattern** (from the famous "Gang of Four" book). However, Python's implementation relies on *functional programming* (closures/higher-order functions) rather than classical object-oriented wrappers.
3. 🚀 **Flask & FastAPI:** The entire routing system of modern web frameworks like Flask (`@app.route('/')`) and FastAPI (`@app.get("/")`) is just **decorators** mapping URLs to functions!
4. 🕵️‍♂️ **The `__closure__` Attribute:** Every function in Python has a `__closure__` attribute. If it's `None`, it's not a closure. If it's a tuple of "cells", it's a closure holding onto enclosing variables!